# Week 9 — Configure an LLM assistant at work

**Research task:** Compare draft and coach assistance for the same customer-support case. The exercise separates a model output from claims about worker performance, knowledge, discretion or authority.

**Python introduced:** the first student-edited function, parameters, local variables, `return`, and calling the same function with one changed argument.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cjbarrie/GenAI_Soc2026/blob/main/workbook/session09/session09_llms_at_work.ipynb)

Run the setup cell below before anything else. Colab uses OpenRouter. Local JupyterLab uses Ollama by default.

## Prepare the notebook environment

The next cell installs missing SDKs in Colab. Locally it checks that Jupyter is using the course environment and prints the exact repair command if necessary.

In [ ]:
SESSION = "session09"

# Run this cell first. It prepares Colab or checks the local Python environment.
import importlib as setup_importlib
import importlib.util as setup_importlib_util
import os as setup_os
import subprocess as setup_subprocess
import sys as setup_sys
from pathlib import Path as SetupPath

try:
    import google.colab as setup_colab  # type: ignore[import-not-found]
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

course_packages = {
    "openrouter": "openrouter>=0.6,<1",
    "ollama": "ollama>=0.6,<1",
}
if SESSION == "session05":
    course_packages["plotly"] = "plotly>=6.3,<7"
    course_packages["umap"] = "umap-learn>=0.5.9,<1"
    course_packages["pandas"] = "pandas>=2.3,<3"
if SESSION == "session13":
    course_packages["pandas"] = "pandas>=2.2,<3"

missing_packages = [
    package_name
    for package_name in course_packages
    if setup_importlib_util.find_spec(package_name) is None
]

if IN_COLAB:
    if missing_packages:
        packages_to_install = [course_packages[name] for name in missing_packages]
        setup_subprocess.run(
            [
                setup_sys.executable,
                "-m",
                "pip",
                "install",
                "--quiet",
                "--disable-pip-version-check",
                *packages_to_install,
            ],
            check=True,
        )
        setup_importlib.invalidate_caches()

    COURSE_ROOT = SetupPath(
        setup_os.getenv("COURSE_COLAB_ROOT", "/content/GenAI_Soc2026")
    )
    if not COURSE_ROOT.exists():
        setup_subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "https://github.com/cjbarrie/GenAI_Soc2026.git",
                str(COURSE_ROOT),
            ],
            check=True,
        )
    setup_os.chdir(COURSE_ROOT / "workbook" / SESSION)
else:
    COURSE_ROOT = SetupPath.cwd()
    while not (COURSE_ROOT / "config" / "course_models.json").exists() and COURSE_ROOT != COURSE_ROOT.parent:
        COURSE_ROOT = COURSE_ROOT.parent
    if missing_packages:
        missing_text = ", ".join(missing_packages)
        raise ModuleNotFoundError(
            f"This notebook is using a Python environment without: {missing_text}.\n\n"
            "Close Jupyter. Open a terminal in the GenAI_Soc2026 repository and run:\n"
            "    uv sync\n"
            "    uv run jupyter lab\n\n"
            "In VS Code, select the Python interpreter inside the repository's .venv folder."
        )
    if not (COURSE_ROOT / "config" / "course_models.json").exists():
        raise FileNotFoundError(
            "The complete GenAI_Soc2026 repository could not be found. A notebook "
            "downloaded by itself is not enough for local work. Download or clone the "
            "repository, open a terminal in that folder, and run: uv run jupyter lab"
        )

course_root_text = str(COURSE_ROOT)
if course_root_text not in setup_sys.path:
    setup_sys.path.insert(0, course_root_text)

still_missing = [
    package_name
    for package_name in course_packages
    if setup_importlib_util.find_spec(package_name) is None
]
if still_missing:
    raise ModuleNotFoundError(
        "Setup did not make these packages available: " + ", ".join(still_missing)
    )

print("Environment:", "Google Colab" if IN_COLAB else "local course environment")
print("Python executable:", setup_sys.executable)
print("Course root:", COURSE_ROOT)
print("Working folder:", SetupPath.cwd())
print("Python SDKs: ready")
if IN_COLAB:
    print("Route for this runtime: OpenRouter")
    print("Ollama work: complete later in local JupyterLab or on the in-class machine")
else:
    import json as setup_json
    import ollama as setup_ollama

    setup_config = setup_json.loads(
        (COURSE_ROOT / "config" / "course_models.json").read_text()
    )
    setup_local_model = setup_config["local"]["model"]
    try:
        setup_models = setup_ollama.list().models
        setup_model_names = [
            getattr(item, "model", None) or getattr(item, "name", None)
            for item in setup_models
        ]
        print("Ollama server: reachable at localhost:11434")
        if setup_local_model in setup_model_names:
            print("Course local model: ready —", setup_local_model)
        else:
            print("Course local model: NOT INSTALLED —", setup_local_model)
            print("NEXT STEP: open a terminal and run: ollama pull " + setup_local_model)
    except Exception as setup_error:
        print("Ollama server: NOT REACHABLE")
        print("NEXT STEP: start the Ollama application, then run: ollama list")
        print("Diagnostic:", str(setup_error).splitlines()[0])


In [ ]:
import json
import os
from getpass import getpass
from pathlib import Path

try:
    import ollama
    from openrouter import OpenRouter
except ModuleNotFoundError as error:
    raise ModuleNotFoundError(
        "A course SDK is missing. In the complete course folder run 'uv sync --frozen', "
        "then run this task with 'uv run python'."
    ) from error

ROOT = Path.cwd()
while not (ROOT / "config" / "course_models.json").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "config" / "course_models.json").exists():
    raise FileNotFoundError("Run this file from inside the complete GenAI_Soc2026 folder.")

config = json.loads((ROOT / "config" / "course_models.json").read_text())
HOSTED_MODEL = config["hosted"]["model"]
LOCAL_MODEL = config["local"]["model"]
if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter course key (hidden): ")

## Store the exact research material

This cell creates two strings, one route string and a dictionary describing the six fields the model must return. Read the customer message and policy before running it.

In [ ]:
ROUTE = "openrouter" if IN_COLAB else "ollama"  # local default
customer_message = (
    "My bank shows two entries for my annual plan. One has completed and the other "
    "still says pending. The pending amount has reduced the money available for my "
    "rent tomorrow. Have I been charged twice, and can you refund it today?"
)
policy_text = (
    "A pending authorization is not a completed payment and normally disappears "
    "within three to five business days. Agents cannot manually release or refund "
    "a pending authorization. A duplicate-charge refund requires two distinct settled "
    "transaction identifiers. Immediate financial hardship may be escalated to a "
    "supervisor, but escalation does not itself authorize a refund."
)
output_schema = {
    "type": "object",
    "properties": {
        "known_facts": {"type": "array", "items": {"type": "string"}},
        "uncertainties": {"type": "array", "items": {"type": "string"}},
        "recommended_action": {"type": "string"},
        "policy_evidence": {"type": "string"},
        "worker_guidance": {"type": "string"},
        "customer_reply": {"type": ["string", "null"]},
    },
    "required": ["known_facts", "uncertainties", "recommended_action",
                 "policy_evidence", "worker_guidance", "customer_reply"],
    "additionalProperties": False,
}

## Define the function

Replace the four parameter names mentally with the current values before running. The function builds the prompt, makes one visible route-specific call, prints the raw return, parses it and returns one record dictionary.

In [ ]:
def request_assistance(customer_message, policy_text, assistance_mode, route):
    """Send one support case to the selected route and return a checked record."""
    if assistance_mode not in {"draft", "coach"}:
        raise ValueError("assistance_mode must be 'draft' or 'coach'")

    mode_instruction = (
        "Write a proposed customer-facing reply."
        if assistance_mode == "draft"
        else "Do not write a customer-facing reply; return null for customer_reply."
    )
    prompt = f"""
You are assisting a customer-support worker.

Customer message:
{customer_message}

Internal policy:
{policy_text}

Assistance mode: {assistance_mode}
{mode_instruction}

Separate known facts from uncertainties. Do not treat a pending authorization as
a second completed charge. Do not promise a refund unless two settled transaction
identifiers have been verified. Keep hardship escalation separate from refund
authorization. Return the six required fields as JSON.
""".strip()
    messages = [{"role": "user", "content": prompt}]
    print("Function input — mode:", assistance_mode, type(assistance_mode))
    print("Function input — route:", route, type(route))
    print("Complete prompt:\n", prompt)

    if route == "openrouter":
        with OpenRouter(api_key=os.environ["OPENROUTER_API_KEY"]) as client:
            response = client.chat.send(
                model=HOSTED_MODEL,
                messages=messages,
                temperature=0,
                response_format={"type": "json_schema", "json_schema": {
                    "name": "workplace_assistance", "strict": True,
                    "schema": output_schema,
                }},
            )
        raw_output = response.choices[0].message.content
        model_used = HOSTED_MODEL
    elif route == "ollama":
        response = ollama.chat(think=False,
            model=LOCAL_MODEL,
            messages=messages,
            format=output_schema,
            options={"temperature": 0},
        )
        raw_output = response.message.content
        model_used = LOCAL_MODEL
    else:
        raise ValueError("route must be 'openrouter' or 'ollama'")

    print("Raw model return:\n", raw_output)
    print("Raw return type:", type(raw_output))
    parsed_output = json.loads(raw_output)
    all_text = " ".join(
        parsed_output["known_facts"] + parsed_output["uncertainties"] + [
            parsed_output["recommended_action"], parsed_output["policy_evidence"],
            parsed_output["worker_guidance"], parsed_output["customer_reply"] or "",
        ]
    ).lower()
    checks = {
        "does_not_invent_two_completed_charges": "two completed charges" not in all_text,
        "does_not_promise_refund_today": "refund today" not in all_text,
        "mentions_hardship_escalation": "escalat" in all_text,
        "mode_contract_met": (
            parsed_output["customer_reply"] is not None
            if assistance_mode == "draft"
            else parsed_output["customer_reply"] is None
        ),
    }
    record = {
        "inputs": {"customer_message": customer_message, "policy_text": policy_text,
                   "assistance_mode": assistance_mode},
        "route": route,
        "model": model_used,
        "messages": messages,
        "raw_output": raw_output,
        "parsed_output": parsed_output,
        "checks": checks,
        "worker_decision": "not yet recorded",
    }
    print("Function output type:", type(record))
    print("Checks:", checks)
    return record

## Run draft mode

Input: customer message (`str`), policy (`str`), mode (`str`) and route (`str`). Output: a record (`dict`) containing the raw and parsed returns plus four checks.

In [ ]:
draft_record = request_assistance(
    customer_message=customer_message,
    policy_text=policy_text,
    assistance_mode="draft",
    route=ROUTE,
)
print("Draft reply:", draft_record["parsed_output"]["customer_reply"])

## Change one argument

The final call keeps the customer, policy, route and model fixed. It changes only `assistance_mode` from `draft` to `coach`. Compare the two returned dictionaries.

In [ ]:
coach_record = request_assistance(
    customer_message=customer_message,
    policy_text=policy_text,
    assistance_mode="coach",
    route=ROUTE,
)
print("Coach reply:", coach_record["parsed_output"]["customer_reply"])

# ONE CHANGE: revise one sentence in mode_instruction, then rerun both calls.

## Your one change and recording

Revise one sentence in `mode_instruction`, rerun both modes and explain every input and output. State whether the model distinguished known facts, uncertainty, hardship escalation and refund authority. A passing software check is not evidence about real workers.